In [10]:
import sys
sys.path.insert(0, "../../run")
from run_config import REPO_PATH, SEASONS, DATA_PROCESSING_PATH
sys.path.insert(1, f"{REPO_PATH}")

import pandas as pd
import joblib
import numpy as np

In [2]:
processed_data_path='../../data/processed/premier_league/'

In [3]:
seasons=sorted(SEASONS)

In [4]:
data_dfs=[pd.read_csv(f"{processed_data_path}/{season}/all_data_df.csv") for season in seasons]

In [7]:
class PreviousSeasonTeamAverager:
    def __init__(self, decay_factor=1.0, date_col='date', home_col='home', away_col='away'):
        self.decay_factor = decay_factor
        self.date_col = date_col
        self.home_col = home_col
        self.away_col = away_col

    def _get_numeric_cols(self, df):
        return [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col]) and col != self.date_col]

    def _compute_team_averages(self, df):
        df = df.copy()
        df[self.date_col] = pd.to_datetime(df[self.date_col])
        most_recent_date = df[self.date_col].max()
        df['days_ago'] = (most_recent_date - df[self.date_col]).dt.days
        df['weights'] = np.exp(-self.decay_factor * df['days_ago'])

        numeric_cols = self._get_numeric_cols(df)
        team_stats = {}

        for team_col in [self.home_col, self.away_col]:
            for team in df[team_col].unique():
                team_matches = df[df[team_col] == team]
                if team_matches.empty:
                    continue
                weighted_sum = (team_matches[numeric_cols].multiply(team_matches['weights'], axis=0)).sum()
                total_weight = team_matches['weights'].sum()
                if total_weight == 0:
                    continue
                avg_stats = (weighted_sum / total_weight).to_dict()
                if team not in team_stats:
                    team_stats[team] = avg_stats
                else:
                    # average if seen as both home and away
                    for k, v in avg_stats.items():
                        team_stats[team][k] = (team_stats[team].get(k, 0) + v) / 2

        return team_stats

    def transform(self, season_dfs):
        season_dfs = sorted(season_dfs, key=lambda df: pd.to_datetime(df[self.date_col].iloc[0]))
        results = []

        for i in range(1, len(season_dfs)):
            prev_season = season_dfs[i - 1]
            current_season = season_dfs[i].copy()
            team_avg_stats = self._compute_team_averages(prev_season)

            for team_role in [self.home_col, self.away_col]:
                role_avg_df = current_season[team_role].map(lambda team: team_avg_stats.get(team, {}))
                role_avg_df = pd.json_normalize(role_avg_df)
                role_avg_df.columns = [f'prev_season_avg_{team_role}_{col}' for col in role_avg_df.columns]

            results.append(role_avg_df)

        return pd.concat(results, ignore_index=True)


In [11]:
previous_season_feature= PreviousSeasonTeamAverager(decay_factor=0.1, date_col='date', home_col='home', away_col='away')
averaged_features = previous_season_feature.transform(data_dfs)

In [12]:
averaged_features

,prev_season_avg_away_home_performance_pk,prev_season_avg_away_home_performance_pkatt,prev_season_avg_away_home_performance_crdr,prev_season_avg_away_home_performance_touches,prev_season_avg_away_home_performance_tkl,prev_season_avg_away_home_performance_int,prev_season_avg_away_home_performance_blocks,prev_season_avg_away_home_expected_xg,prev_season_avg_away_home_expected_npxg,prev_season_avg_away_home_expected_xag,...,prev_season_avg_away_away_performance_tklw,prev_season_avg_away_away_performance_pkwon,prev_season_avg_away_away_performance_pkcon,prev_season_avg_away_away_performance_og,prev_season_avg_away_away_performance_recov,prev_season_avg_away_away_aerial_duels_won,prev_season_avg_away_away_aerial_duels_lost,prev_season_avg_away_away_aerial_duels_won%,prev_season_avg_away_days_ago,prev_season_avg_away_weights
0,8.199048e-01,0.819905,2.795604e-07,676.577433,21.328001,11.849133,13.856124,1.978975,1.324568,0.735056,...,11.260321,0.000737,0.819905,0.000000e+00,55.261639,13.827141,13.357238,51.457340,5.910794,0.682140
1,1.646321e-01,0.329219,4.958285e-02,636.563947,13.753811,8.383517,6.708493,1.507748,1.244376,0.957986,...,6.850128,0.392395,0.329219,1.106118e-02,39.709626,11.361316,11.215092,51.575188,6.388455,0.664876
2,5.240951e-02,0.052599,2.085561e-02,554.781929,11.306244,10.419716,11.984419,1.234291,1.192195,0.647071,...,9.774098,0.259471,0.052599,1.092958e-02,51.575712,17.012424,18.465820,49.128133,6.261785,0.662801
3,5.324300e-02,0.053243,3.710568e-08,560.336914,15.366622,8.907767,10.090019,0.893745,0.851151,0.580526,...,8.852333,0.002715,0.053243,8.343855e-12,59.997358,31.328390,27.885370,53.095282,6.273386,0.656664
4,3.191396e-04,0.000319,1.327431e-07,585.645509,16.809178,8.409734,12.774007,1.365427,1.365171,1.071041,...,10.311476,0.094100,0.000319,1.076143e-09,45.168508,17.964677,16.092220,52.949098,6.473899,0.631748
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1505,1.856759e-01,0.185676,4.029989e-10,739.488577,20.949867,7.298129,11.404949,2.943703,2.803969,1.778957,...,13.867140,0.000011,0.185676,5.794840e-04,43.495767,11.222432,13.593203,39.056742,9.296564,0.572086
1506,3.639715e-07,0.060968,1.408099e-04,660.863013,14.234198,8.045015,11.922550,1.922073,1.867202,1.656119,...,12.826647,0.003395,0.060968,2.254509e-04,38.703021,6.122694,10.215260,38.791717,8.722171,0.551881
1507,3.936346e-01,0.393635,3.528292e-01,526.993156,15.818160,6.781692,12.525956,1.087237,0.772329,0.579428,...,8.063951,0.352829,0.393635,3.248112e-04,41.462948,9.670097,9.814879,49.314103,8.718824,0.584868
1508,1.856759e-01,0.185676,4.029989e-10,739.488577,20.949867,7.298129,11.404949,2.943703,2.803969,1.778957,...,13.867140,0.000011,0.185676,5.794840e-04,43.495767,11.222432,13.593203,39.056742,9.296564,0.572086
